# DigiStructMed — Full KG Construction Pipeline

**Neurosymbolic pipeline · SHACL-validated**

---

| Step | Name | Type |
|------|------|------|
| 1a | Text extraction | Tool |
| 1b | Table extraction | Tool |
| 1c | NER (BERT) | 🔵 Neural |
| 1d | Entity Linking (UMLS BK) | 🟢 Symbolic |
| 1e | LLM Disambiguation (conditional) | 🔵 Neural |
| 2  | Ontology Loading | 🟢 Symbolic |
| 3a | Table → RML Mappings | 🟢 Symbolic |
| 3b | Text Path (v1 \| v2) | ⚠️ Open question |
| 4  | KG Materialization (Morph-KGC) | 🟢 Symbolic |
| 5  | SHACL Validation (TravSHACL / pySHACL fallback) | 🟢 Symbolic |

> **Non-negotiable invariant:** No raw LLM output enters the KG directly.  
> All assertions → structured JSON → RML → engine → RDF.

---
## 0. Setup

In [ ]:
# Install all pipeline dependencies
!pip install -q pymupdf pdfplumber rdflib transformers accelerate sentencepiece
!pip install -q morph-kgc

# SHACL validation
# TravSHACL — validates via SPARQL endpoint when SPARQL_ENDPOINT is set in config
#   https://github.com/SDM-TIB/Trav-SHACL
!pip install -q travshacl
# pySHACL — automatic fallback when no SPARQL endpoint is configured (works in Colab)
!pip install -q pyshacl

# docling — only needed when EXTRACTION_VERSION = 'v2' (step1a + step1b)
!pip install -q docling

# Hugging Face Hub — Inference API + gated model downloads
!pip install -q huggingface_hub

# Step 1d entity linking — fast fuzzy ratios over candidate sets (fallback: difflib)
!pip install -q rapidfuzz

In [ ]:
import os, sys, zipfile
from pathlib import Path
from typing import Optional

# True = show Colab's **Choose Files** button and upload the project ZIP.
# False = skip (use this if you already unzipped or `git clone`d in /content).
UPLOAD_PROJECT_ZIP = True


def _find_project_with_scripts(here: Path) -> Optional[Path]:
    """Return a directory that contains both scripts/ and the notebook."""
    if (here / 'scripts').is_dir():
        return here.resolve()
    for child in sorted(here.iterdir()):
        if child.is_dir() and (child / 'scripts').is_dir():
            return child.resolve()
    return None


root = Path.cwd().resolve()

if UPLOAD_PROJECT_ZIP:
    try:
        from google.colab import files
        print(
            'Click **Choose Files** below, then upload DigiStructMed_thesis_colab.zip '
            '(from create_colab_zip.py on your PC).'
        )
        uploaded = files.upload()
        for name, data in uploaded.items():
            dest = root / name
            dest.write_bytes(data)
            print(f'Received {len(data) // 1024} KB → {name}')
            if name.lower().endswith('.zip'):
                with zipfile.ZipFile(dest, 'r') as zf:
                    zf.extractall(root)
                print('ZIP extracted.')
            break
    except ImportError:
        print('Not running in Colab — using current folder (no upload).')

proj = _find_project_with_scripts(root)
if proj is not None and proj != root:
    os.chdir(proj)
    root = Path.cwd().resolve()
    print(f'Using project folder: {root}')

if not (root / 'scripts').is_dir():
    raise RuntimeError(
        'Cannot find scripts/ — upload the thesis Colab ZIP above, or set '
        'UPLOAD_PROJECT_ZIP = False and %cd into the repo root first.'
    )

scripts_dir = str(root / 'scripts')
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

print('Working directory:', root)
print('scripts/ on path:', scripts_dir)

---
## 0b. Upload inputs (Colab)

Run the next cell to **create `input/`** and **upload from your computer**:
- guideline PDF  
- ontology (`.ttl` T-Box)  
- UMLS subset CSV for Step 1d entity linking (same pattern as PDF/ontology — use `UPLOAD_UMLS = False` only if you already copied the file to `input/` and will set `UMLS_CSV_PATH` in the config cell)

Set the `UPLOAD_*` flags to `False` to skip that upload prompt and use files you already placed under `input/`.

If you are **not** on Colab, skip this cell and copy files into `input/` yourself (then set `UMLS_CSV_PATH` in the config cell if the path is not the default).

In [ ]:
from pathlib import Path

INPUT_DIR = Path('input')
INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to False to skip that upload dialog (use a file you already placed under input/)
UPLOAD_PDF       = True
UPLOAD_ONTOLOGY  = True
UPLOAD_UMLS      = True    # Step 1d — set False only if you set UMLS_CSV_PATH manually in config

# Canonical paths used by the rest of the notebook
PDF_PATH       = str(INPUT_DIR / 'guideline.pdf')
ONTOLOGY_PATH  = str(INPUT_DIR / 'hf_guideline_ontology.ttl')
UMLS_CSV_PATH  = None

try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    if UPLOAD_PDF:
        print('— Upload your guideline PDF (one file) —')
        up = files.upload()
        for name, data in up.items():
            Path(PDF_PATH).write_bytes(data)
            print(f'Saved: {PDF_PATH}  (from {name})')
            break
    if UPLOAD_ONTOLOGY:
        print('— Upload your ontology file (.ttl) —')
        up = files.upload()
        for name, data in up.items():
            Path(ONTOLOGY_PATH).write_bytes(data)
            print(f'Saved: {ONTOLOGY_PATH}  (from {name})')
            break
    if UPLOAD_UMLS:
        print('— Upload your UMLS CSV (cui, label, …) —')
        up = files.upload()
        for name, data in up.items():
            dest = INPUT_DIR / 'umls_subset.csv'
            dest.write_bytes(data)
            UMLS_CSV_PATH = str(dest)
            print(f'Saved: {UMLS_CSV_PATH}  (from {name})')
            break
else:
    print('Not on Colab — put files at:')
    print(' ', PDF_PATH)
    print(' ', ONTOLOGY_PATH)
    print(' ', INPUT_DIR / 'umls_subset.csv', '(or set UMLS_CSV_PATH in the config cell)')

print('\nPaths for next cell:')
print('  PDF_PATH      ', PDF_PATH)
print('  ONTOLOGY_PATH ', ONTOLOGY_PATH)
print('  UMLS_CSV_PATH ', UMLS_CSV_PATH)

---
## 0c. Hugging Face token (interactive)

Step **1e** always loads the LM locally (Llama via Hugging Face Hub); set **HF_TOKEN** for gated models. Step **3b** (v2) also respects `LLM_BACKEND` (`hf_local` or `hf_inference`). Run the next cell and **paste your token** when prompted, or use Colab Secrets.

Create a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). This notebook does **not** read `HF_TOKEN` from the environment unless you uncomment the fallback in the config cell.

In [ ]:
from getpass import getpass

print(
    'Hugging Face token (hf_...): needed for gated model download and HF Inference API.\n'
    'Leave empty if you will not use HF (Step 1e / 3b v2 will skip LLM).'
)
_hf = getpass('HF_TOKEN: ')
HF_TOKEN = _hf.strip() or None
if HF_TOKEN:
    print('HF_TOKEN set (length', len(HF_TOKEN), 'chars).')
else:
    print('HF_TOKEN not set.')

In [ ]:
# ── Pipeline configuration — paths / keys ────────────────────────────────
# PDF_PATH, ONTOLOGY_PATH, UMLS_CSV_PATH are set in cell "0b. Upload inputs".
# HF_TOKEN is set in cell "0c. Hugging Face token".
# If you skipped those cells, define paths / keys here:
try:
    PDF_PATH
except NameError:
    PDF_PATH = 'input/guideline.pdf'
try:
    ONTOLOGY_PATH
except NameError:
    ONTOLOGY_PATH = 'input/hf_guideline_ontology.ttl'
try:
    UMLS_CSV_PATH
except NameError:
    UMLS_CSV_PATH = None

# If you set UPLOAD_UMLS = False in 0b, set your local UMLS CSV path here:
# UMLS_CSV_PATH = 'input/umls_subset.csv'

try:
    HF_TOKEN
except NameError:
    HF_TOKEN = None

# Uncomment only if you intentionally want to reuse HF_TOKEN from the process environment:
# import os
# HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN')

# 3B fits a Colab T4 (~16GB VRAM); 8B is tighter on T4 — use 8B only if you have headroom.
HF_MODEL        = 'meta-llama/Llama-3.2-3B-Instruct'

# LLM backend for Step 1e + Step 3b (v2) — Hugging Face Llama only:
#   'hf_local'     — load weights on this runtime (needs GPU, e.g. T4; HF_TOKEN for gated Llama)
#   'hf_inference' — HF Serverless API (uses inference credits; use on CPU-only if you pay / have quota)
#   'none'         — skip LLM (1e skips; 3b v2 falls back to v1 rules)
if HF_TOKEN:
    LLM_BACKEND = 'hf_local'   # default: no HF Inference API bill; needs GPU (Runtime → GPU)
else:
    LLM_BACKEND = 'none'

# CPU-only Colab or you prefer hosted API (paid / credit-limited):
# LLM_BACKEND = 'hf_inference'

# Extraction version — controls step1a (text) AND step1b (tables)
# 'v1' : PyMuPDF text  + pdfplumber tables  (fast, no extra deps)
# 'v2' : Docling text  + Docling tables      (better for complex layouts)
# Use matching versions for both steps so text and tables come from the same parser.
EXTRACTION_VERSION = 'v2'   # 'v1' | 'v2'
SKIP_FIRST_PAGES   = 3      # cover / copyright pages
SKIP_LAST_PAGES    = 5      # reference / index pages

TEXT_PATH_VERSION = 'v2'  # 'v1' | 'v2'
                           # run both below and compare SHACL metrics

NER_MODEL       = 'd4data/biomedical-ner-all'

print(f'PDF             : {PDF_PATH}')
print(f'Ontology        : {ONTOLOGY_PATH}')
print(f'UMLS CSV        : {UMLS_CSV_PATH}')
print(f'LLM backend     : {LLM_BACKEND}  (HF_MODEL={HF_MODEL})')
print(f'Extraction ver  : {EXTRACTION_VERSION}  (step1a + step1b)')
print(f'Text path ver   : {TEXT_PATH_VERSION}  (step3b)')

---
## Step 1 — Source Structuring

Produces **S** = `{ text_blocks.json, table_index.json, tables/*.csv, resolved_entities.json }`

In [ ]:
# ── 1a · Text extraction ──────────────────────────────────────────────────
from step1a_extract_text import extract_text

text_blocks = extract_text(
    pdf_path=PDF_PATH,
    output_dir='outputs/step1',
    skip_first_pages=SKIP_FIRST_PAGES,
    skip_last_pages=SKIP_LAST_PAGES,
    version=EXTRACTION_VERSION,
)
print(f'\n→ [{EXTRACTION_VERSION}] {len(text_blocks)} text blocks')

In [ ]:
# ── 1b · Table extraction ─────────────────────────────────────────────────
from step1b_extract_tables import extract_tables

# version must match step1a: v1=pdfplumber, v2=Docling
table_index = extract_tables(
    pdf_path=PDF_PATH,
    output_dir='outputs/step1',
    version=EXTRACTION_VERSION,
    skip_first_pages=SKIP_FIRST_PAGES,
    skip_last_pages=SKIP_LAST_PAGES,
)
print(f'\n→ [{EXTRACTION_VERSION}] {len(table_index)} tables extracted')

In [ ]:
# ── 1c · NER  [NEURAL — BERT] ────────────────────────────────────────────
from step1c_ner import run_ner

mentions = run_ner(
    text_blocks_path='outputs/step1/text_blocks.json',
    output_dir='outputs/step1',
    model_name=NER_MODEL,
    min_score=0.55,
)
print(f'\n→ {len(mentions)} entity mentions')

In [ ]:
# ── 1d · Entity Linking  [SYMBOLIC — rules + UMLS BK] ────────────────────
from step1d_entity_linking import link_entities

grounded = link_entities(
    mentions_path='outputs/step1/entity_mentions.json',
    umls_csv_path=UMLS_CSV_PATH,
    output_dir='outputs/step1',
    top_k=15,
    sim_threshold=0.96,
)

In [ ]:
# ── 1e · LLM Disambiguation  [NEURAL — conditional] ──────────────────────
# Invoked ONLY for entities with linking_status == 'needs_disambiguation'
from step1e_disambiguate import disambiguate

resolved = disambiguate(
    grounded_path='outputs/step1/grounded_entities.json',
    output_dir='outputs/step1',
    hf_token=HF_TOKEN,
    hf_model=HF_MODEL,
)
linked_count = sum(1 for e in resolved if e.get('cui_final'))
print(f'\n→ {linked_count}/{len(resolved)} entities have a final CUI')

---
## Step 2 — Ontology Loading

Produces **O** = `OntologyIndex` (classes, properties, restrictions, enumerations)

In [ ]:
from step2_load_ontology import load_ontology

ontology = load_ontology(ONTOLOGY_PATH)

print(f'\nOntology summary:')
print(f'  Classes           : {len(ontology.classes)}')
print(f'  Object properties : {len(ontology.object_properties)}')
print(f'  Datatype properties: {len(ontology.datatype_properties)}')
print(f'  Named individuals : {len(ontology.named_individuals)}')
print(f'  Enumerations      : {len(ontology.enumerations)}')

---
## Step 3 — Mapping Generation

Produces **M** = `{ table_mappings.ttl, text_mappings_<v>.ttl }`

In [ ]:
# ── 3a · Table path  [SYMBOLIC — deterministic] ───────────────────────────
# Fuzzy match column headers → ontology properties + review gate
from step3a_table_mappings import generate_table_mappings

table_mapping_result = generate_table_mappings(
    table_index_path='outputs/step1/table_index.json',
    ontology=ontology,
    output_dir='outputs/step3',
)
print('\nTable mapping counts:', table_mapping_result['counts'])
print('Review file (check before Step 4):', table_mapping_result['review_path'])

In [ ]:
# ── 3b · Text path  [OPEN QUESTION — versioned] ──────────────────────────
#  v1 — co-occurrence + type rules  |  v2 — LLM SPO + ontology gate
# Run BOTH below, then compare SHACL results in Step 5.
from step3b_text_path import run_text_path

text_result = run_text_path(
    resolved_entities_path='outputs/step1/resolved_entities.json',
    text_blocks_path='outputs/step1/text_blocks.json',
    ontology=ontology,
    output_dir='outputs/step3',
    version=TEXT_PATH_VERSION,
    llm_backend=LLM_BACKEND,
    hf_token=HF_TOKEN,
    hf_model=HF_MODEL,
)
print(f"\n→ [{TEXT_PATH_VERSION}] {text_result['count']} text assertions")

---
## Step 4 — KG Materialization

Executes M on S → RDF A-Box, merges T-Box, adds CUI instance triples.

Output: `outputs/step4/output_<version>.ttl`

In [ ]:
from step4_materialize import materialize

kg_path = materialize(
    table_mappings_path='outputs/step3/table_mappings.ttl',
    text_mappings_path=f'outputs/step3/text_mappings_{TEXT_PATH_VERSION}.ttl',
    text_assertions_path=f'outputs/step3/text_assertions_{TEXT_PATH_VERSION}.json',
    resolved_entities_path='outputs/step1/resolved_entities.json',
    ontology_path=ONTOLOGY_PATH,
    output_dir='outputs/step4',
    version=TEXT_PATH_VERSION,
)
print(f'\n→ KG written to: {kg_path}')

---
## Step 5 — SHACL Validation

Validates A-Box against shapes derived from the OWL ontology.

Output: `outputs/step5/validation_report_<version>.json`

In [ ]:
from step5_validate import validate

# ── TravSHACL vs pySHACL ─────────────────────────────────────────────────
# If you have a SPARQL endpoint running (e.g. Apache Jena Fuseki with the KG
# loaded), set SPARQL_ENDPOINT to its URL and TravSHACL will be used.
# Leave as None to use pySHACL (no server needed — works in Colab).
SPARQL_ENDPOINT = None   # e.g. 'http://localhost:3030/kg/sparql'

report = validate(
    kg_path=f'outputs/step4/output_{TEXT_PATH_VERSION}.ttl',
    ontology_path=ONTOLOGY_PATH,
    output_dir='outputs/step5',
    version=TEXT_PATH_VERSION,
    sparql_endpoint=SPARQL_ENDPOINT,
)

print(f"\nBackend         : {report['backend']}")
print(f"Conforms        : {report['conforms']}")
print(f"Total triples   : {report['total_triples']}")
print(f"Violations      : {report['total_violations']}")
print(f"Conformance     : {report['metrics']['conformance_ratio']:.1%}")

---
## Run Both Versions & Compare

This resolves the open question empirically.  
The version with higher `conformance_ratio` and higher `total_triples` wins.

In [ ]:
from step3b_text_path import run_text_path
from step4_materialize import materialize
from step5_validate   import validate, compare_versions

for ver in ('v1', 'v2'):
    print(f'\n{'='*50}')
    print(f'  Running version: {ver}')
    print('='*50)

    run_text_path(
        resolved_entities_path='outputs/step1/resolved_entities.json',
        text_blocks_path='outputs/step1/text_blocks.json',
        ontology=ontology,
        output_dir='outputs/step3',
        version=ver,
        llm_backend=LLM_BACKEND,
        hf_token=HF_TOKEN,
        hf_model=HF_MODEL,
    )
    materialize(
        table_mappings_path='outputs/step3/table_mappings.ttl',
        text_mappings_path=f'outputs/step3/text_mappings_{ver}.ttl',
        text_assertions_path=f'outputs/step3/text_assertions_{ver}.json',
        resolved_entities_path='outputs/step1/resolved_entities.json',
        ontology_path=ONTOLOGY_PATH,
        output_dir='outputs/step4',
        version=ver,
    )
    validate(
        kg_path=f'outputs/step4/output_{ver}.ttl',
        ontology_path=ONTOLOGY_PATH,
        output_dir='outputs/step5',
        version=ver,
    )

compare_versions()

---
## Download Results

In [ ]:
import shutil
from pathlib import Path

# Pack all outputs into a ZIP for download
shutil.make_archive('DigiStructMed_outputs', 'zip', 'outputs')

try:
    from google.colab import files
    files.download('DigiStructMed_outputs.zip')
except ImportError:
    print('Not running in Colab — outputs ZIP is at: DigiStructMed_outputs.zip')